# Midwicket Quickstart

**Midwicket** is a Python library for agentic cricket analytics.

This notebook covers the two primary APIs:

- **Win Probability** — single match-state prediction
- **Win Probability Timeline** — over-by-over simulation

> All examples run entirely in memory. No dataset download is required.

In [ ]:
# Step 1: Install Midwicket
# Typical install time: ~20 seconds on Colab.
!pip install midwicket -q

In [ ]:
# Step 2: Verify installation
import midwicket
print(f"Midwicket {midwicket.__version__} loaded successfully.")

---
## Win Probability — Single Prediction

`px.predict_win()` returns a win probability estimate from any match state.
It uses a lightweight in-memory model — no external data required.

In [ ]:
import midwicket.express as px

# Scenario: chasing 180, currently 120/5 after 15 overs at Wankhede Stadium
result = px.predict_win(
    venue="Wankhede Stadium",
    target=180,
    current_score=120,
    wickets_down=5,
    overs_done=15.0,
)

print("Scenario : Chasing 180 | 120/5 | 15 overs completed")
print(f"Win Probability  : {result['win_prob']:.1%}")
print(f"Confidence Score : {result['confidence']:.1%}")

---
## Win Probability Timeline — Over-by-Over

Simulate how win probability shifts across a full chase using in-memory
match state snapshots. No file I/O or network requests involved.

In [ ]:
import midwicket.express as px

# Synthetic match states: (overs_done, current_score, wickets_down)
MATCH_STATES = [
    (0,   0,  0),
    (2,  18,  0),
    (4,  32,  1),
    (6,  55,  1),
    (8,  72,  2),
    (10, 95,  2),
    (12, 110, 3),
    (14, 118, 4),
    (16, 130, 5),
    (18, 152, 6),
    (20, 165, 8),
]

TARGET = 180
VENUE  = "Wankhede Stadium"

timeline = []
for overs, score, wkts in MATCH_STATES:
    r = px.predict_win(
        venue=VENUE,
        target=TARGET,
        current_score=score,
        wickets_down=wkts,
        overs_done=overs,
    )
    timeline.append((overs, score, wkts, r["win_prob"]))

print(f"{'Over':>5}  {'Score':>8}  {'Win Prob':>10}  {'Bar'}")
print("-" * 50)
for overs, score, wkts, wp in timeline:
    bar = "|" * int(wp * 25)
    print(f"{overs:>5}  {score:>4}/{wkts:<3}  {wp:>8.1%}  {bar}")

---
## Win Probability Chart

Visualize the timeline using matplotlib, which is available in Colab by default.

In [ ]:
try:
    import matplotlib.pyplot as plt

    overs_list = [t[0] for t in timeline]
    wp_list    = [t[3] * 100 for t in timeline]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.fill_between(overs_list, wp_list, alpha=0.12, color="#4A90D9")
    ax.plot(overs_list, wp_list, marker="o", linewidth=2.5,
            color="#4A90D9", markerfacecolor="white", markersize=7)
    ax.axhline(50, color="gray", linestyle="--", linewidth=1, label="50% mark")
    ax.set_title(
        f"Win Probability Timeline  |  Chasing {TARGET} at {VENUE}",
        fontsize=13, fontweight="bold"
    )
    ax.set_xlabel("Overs Completed", fontsize=11)
    ax.set_ylabel("Win Probability (%)", fontsize=11)
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 100)
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available. Install with: pip install matplotlib")

---
## Further Reading

| Feature | Usage |
|---|---|
| Full historical IPL dataset | `from midwicket.data.loader import DataLoader; DataLoader().download()` |
| Player statistics | `px.get_player_stats("V Kohli")` |
| Head-to-head matchups | `px.get_matchup("V Kohli", "JJ Bumrah")` |
| Full session with all matches | `px.quick_load()` |

**Source and documentation:** https://github.com/CodersAcademy006/Midwicket